# MQAR: State Size vs. Recall Accuracy (Transformer / Attention)

This is a **single-file, Colab-importable** recreation of the central experiment from
**Zoology** (Arora, Eyuboglu, et al., *"Zoology: Measuring and Improving Recall in
Efficient Language Models"*, https://github.com/HazyResearch/zoology).

We reproduce the **state-size vs. recall-accuracy** plot *for the transformer
(attention) architecture* on the **Multi-Query Associative Recall (MQAR)** task.

### What it does
* Trains one decoder-only transformer per model width in `D_MODELS = [8, 16, 64, 128, 192]`.
* Each width maps to a point on the **x-axis (state size)**. For attention the *state
  size* is the KV-cache size; following zoology's `model.py`, it is
  `4 · Σ_layers (2 · d_model · seq_len)` bytes. With the task difficulty
  (`n_layers`, `seq_len`) fixed, `d_model` is the knob that moves us along the x-axis —
  exactly the knob zoology uses to sweep model state.
* Every run is **fully reproducible** (global determinism + per-run seeds) and the
  **complete nested config is saved to Weights & Biases** (`wandb.init(config=...)`
  *and* a `config.json` artifact), together with library/GPU versions and a dataset
  fingerprint.
* A final summary run logs the recreated curve as a matplotlib figure, a `wandb.Table`,
  and a native W&B line plot.

### Faithfulness to zoology
The MQAR data generator is copied (near-verbatim) from
`zoology/data/multiquery_ar.py`; the model mirrors `zoology/model.py` with
`block_type="TransformerBlock"`, `sequence_mixer=zoology.mixers.attention.MHA`, and
`state_mixer=zoology.mixers.mlp.MLP`; the state-size formula mirrors
`LanguageModel.state_size` / `MHA.state_size`; the training loop mirrors
`zoology/train.py`. We reimplement (rather than `pip install zoology`) only because the
package hard-requires CUDA build deps (`causal_conv1d`, `flash-linear-attention`) that
are fragile in Colab and irrelevant to a pure-attention experiment.

### How to run in Colab
1. Set the runtime to **GPU** (Runtime → Change runtime type → GPU). It also runs on
   CPU, just slower.
2. Set `WANDB_ENTITY` / `WANDB_PROJECT` in the **Config** cell (or leave
   `WANDB_MODE="online"` and log in when prompted; set `WANDB_MODE="disabled"` to run
   without W&B).
3. Run all cells top to bottom.

## 0. Dependencies
Torch / numpy / matplotlib / einops / tqdm ship with Colab; we only ensure `wandb`
(and `einops`, just in case). Using `subprocess` instead of `!pip` keeps this a valid
`.py` file that also works as a plain script.

In [ ]:
import importlib
import subprocess
import sys


def _ensure(pip_name: str, import_name: str = None):
    try:
        importlib.import_module(import_name or pip_name)
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)


for _pkg, _imp in [("wandb", "wandb"), ("einops", "einops"), ("matplotlib", "matplotlib"), ("tqdm", "tqdm")]:
    _ensure(_pkg, _imp)

## 1. Configuration
All experiment knobs live here. The x-axis (state size) range, the number of points,
the (fixed) MQAR difficulty, and the training hyper-parameters are exposed as plain
constants so they are easy to edit in Colab.

In [ ]:
import os
import json
import math
import random
import platform
import hashlib
import uuid
from dataclasses import dataclass, field, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
from tqdm.auto import tqdm

# ---- Weights & Biases -------------------------------------------------------------
WANDB_PROJECT = "zoology-mqar-transformer"
WANDB_ENTITY = None          # <-- set to your W&B entity (username/team), or leave None
WANDB_MODE = "online"        # "online" | "offline" | "disabled"

# ---- The sweep: explicit model widths ---------------------------------------------
# One run per d_model. Each width maps to a state size (the x-axis) via
# attention_state_size(): state = 4 * n_layers * 2 * d_model * seq_len. With the default
# task (n_layers=2, seq_len=128) these widths span state sizes ~1.6e4 -> ~3.9e5.
D_MODELS = [8, 16, 64, 128, 192]
N_POINTS = len(D_MODELS)

# ---- Per-run learning rate: clamped 1/width scaling -------------------------------
# Wider models (larger KV-cache / state) get a proportionally lower peak LR. This is
# what stops the largest model from overshooting into a memorizing (overfitting)
# solution. Anchored so the mid-scale run keeps the LR that was known to work well.
LR_REF = 1e-3                # reference peak LR ...
D_REF = 49                   # ... at this reference model width (the mid-scale run)
LR_MIN = 1e-4                # clamp floor: keep the widest models from going too slow
LR_MAX = 1.5e-3              # clamp ceiling: keep the narrowest models from running hot

# ---- Global reproducibility seed --------------------------------------------------
SEED = 123


@dataclass
class MQARTaskConfig:
    """Fixed MQAR difficulty (mirrors a canonical zoology train/test segment)."""
    vocab_size: int = 8192
    input_seq_len: int = 128
    num_kv_pairs: int = 8
    power_a: float = 0.01
    random_non_queries: bool = True
    num_train_examples: int = 50_000
    num_test_examples: int = 1_000


@dataclass
class ModelConfig:
    """Decoder-only transformer (zoology block_type='TransformerBlock')."""
    d_model: int = 128
    n_layers: int = 2
    num_heads: int = 1
    mlp_hidden_mult: int = 4
    vocab_size: int = 8192
    max_position_embeddings: int = 128
    resid_dropout: float = 0.0
    embed_dropout: float = 0.0
    attn_dropout: float = 0.0
    layer_norm_epsilon: float = 1e-5
    initializer_range: float = 0.02
    learnable_word_embeddings: bool = True
    block_type: str = "TransformerBlock"
    sequence_mixer: str = "zoology.mixers.attention.MHA"
    state_mixer: str = "zoology.mixers.mlp.MLP"


@dataclass
class TrainParams:
    """Training hyper-parameters (zoology/train.py + warmup, grad-clip, patience).

    The peak learning rate is *not* fixed here — it is set per run by `lr_for_d_model`
    (clamped 1/width). The schedule is a linear warmup over `warmup_epochs` followed by
    per-step cosine decay to 0.
    """
    max_epochs: int = 32
    weight_decay: float = 0.1
    batch_size: int = 256
    test_batch_size: int = 256
    warmup_epochs: float = 1.0           # linear LR warmup over this many epochs
    grad_clip: float = 1.0               # max gradient norm (<= 0 disables clipping)
    early_stopping_metric: str = "valid/accuracy"
    early_stopping_threshold: float = 0.99   # stop early once the task is solved
    patience: int = 5                   # stop if valid/accuracy hasn't improved for N epochs
    seed: int = SEED


def lr_for_d_model(d_model: int) -> float:
    """Clamped 1/width peak LR: lr = LR_REF * (D_REF / d_model), clamped to [LR_MIN, LR_MAX]."""
    return float(min(LR_MAX, max(LR_MIN, LR_REF * (D_REF / d_model))))


TASK = MQARTaskConfig()
TRAIN = TrainParams()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: no GPU detected — runs will be slow. In Colab: Runtime > Change runtime type > GPU.")


def set_determinism(seed: int):
    """Make a run reproducible end-to-end (mirrors zoology.utils.set_determinism + CUDA)."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

## 2. MQAR data generation
Copied (near-verbatim) from `zoology/data/multiquery_ar.py`. Each example places
`num_kv_pairs` key→value bindings up front, then queries a subset of those keys; the
model must recall the matching value. Non-query positions are filled with random
distractor tokens. Labels are `-100` (ignored by the loss/metric) except at answer
positions. The large vocabulary (8192) is, per the paper, important for separating
architectures.

In [ ]:
def multiquery_ar(
    vocab_size: int,
    num_examples: int,
    input_seq_len: int,
    seed: int,
    power_a: float = 0.01,
    num_kv_pairs: int = 8,
    num_passes: int = 1,
    random_non_queries: bool = True,
):
    """Generate (inputs, labels) for the multi-query associative recall task.

    Verbatim port of zoology.data.multiquery_ar.multiquery_ar (HazyResearch/zoology).
    """
    assert input_seq_len % 2 == 0, "input_seq_len must be even"
    assert vocab_size > input_seq_len
    assert num_kv_pairs * 2 * num_passes + num_kv_pairs * 2 <= input_seq_len

    np.random.seed(seed)

    context_size = num_kv_pairs * 2 * num_passes

    # keys / values are drawn from disjoint halves of the vocabulary
    key_vocab_size = vocab_size // 2
    key_choices = np.arange(1, key_vocab_size)
    value_choices = np.arange(key_vocab_size, vocab_size)

    keys_unshuffled = np.tile(key_choices, (num_examples, 1))
    keys = np.apply_along_axis(np.random.choice, 1, keys_unshuffled, replace=False, size=num_kv_pairs)

    values_unshuffled = np.tile(value_choices, (num_examples, 1))
    values = np.apply_along_axis(np.random.choice, 1, values_unshuffled, replace=False, size=num_kv_pairs)

    kvs = np.zeros((num_examples, context_size), dtype=np.int64)
    kvs[:, 0::2] = keys
    kvs[:, 1::2] = values
    kvs = np.tile(kvs, (1, num_passes))

    # power-law gap distribution between a key-value pair and its query
    space = (input_seq_len - context_size) // 2
    p = power_a * np.arange(1, space + 1) ** (power_a - 1)
    p = p / p.sum()

    x = np.stack([np.arange(space, dtype=int)] * num_examples)
    gaps = np.apply_along_axis(np.random.choice, axis=1, arr=x, replace=False, p=p, size=num_kv_pairs)

    queries = np.zeros((num_examples, input_seq_len - context_size + 1), dtype=np.int64)
    np.put_along_axis(queries, (gaps * 2), values=keys, axis=1)
    examples = np.concatenate([kvs, queries], axis=1)

    labels = np.full((num_examples, input_seq_len + 1), -100, dtype=np.int64)
    np.put_along_axis(labels, (gaps * 2) + context_size + 1, values=values, axis=1)

    inputs, labels = torch.tensor(examples[:, :-1]), torch.tensor(labels[:, 1:])

    if random_non_queries:
        inputs[inputs == 0] = torch.randint(vocab_size, size=inputs.shape)[inputs == 0]

    return inputs, labels


def build_dataloaders(task: MQARTaskConfig, seed: int):
    """Build reproducible train/test loaders with disjoint train/test seeds."""
    MAX_SEED = 2 ** 32
    rng = np.random.RandomState(seed)
    train_seed = int(rng.randint(0, MAX_SEED // 2))
    test_seed = int(rng.randint(MAX_SEED // 2, MAX_SEED))

    train_inputs, train_labels = multiquery_ar(
        vocab_size=task.vocab_size, num_examples=task.num_train_examples,
        input_seq_len=task.input_seq_len, seed=train_seed,
        power_a=task.power_a, num_kv_pairs=task.num_kv_pairs,
        random_non_queries=task.random_non_queries,
    )
    test_inputs, test_labels = multiquery_ar(
        vocab_size=task.vocab_size, num_examples=task.num_test_examples,
        input_seq_len=task.input_seq_len, seed=test_seed,
        power_a=task.power_a, num_kv_pairs=task.num_kv_pairs,
        random_non_queries=task.random_non_queries,
    )

    # fingerprint the test set so dataset identity is verifiable across machines
    fingerprint = hashlib.md5(test_inputs.numpy().tobytes() + test_labels.numpy().tobytes()).hexdigest()

    train_ds = torch.utils.data.TensorDataset(train_inputs, train_labels)
    test_ds = torch.utils.data.TensorDataset(test_inputs, test_labels)

    g = torch.Generator().manual_seed(seed)  # reproducible shuffling
    train_dl = torch.utils.data.DataLoader(train_ds, batch_size=TRAIN.batch_size, shuffle=True, generator=g)
    test_dl = torch.utils.data.DataLoader(test_ds, batch_size=TRAIN.test_batch_size, shuffle=False)
    return train_dl, test_dl, fingerprint

## 3. Model — decoder-only transformer
A faithful re-implementation of zoology's `LanguageModel` with
`block_type="TransformerBlock"`:
token + learnable absolute position embeddings → `n_layers` pre-norm blocks of
(`MHA` sequence-mixer + `MLP` state-mixer) → final LayerNorm → weight-tied LM head.
Initialization (`std=0.02`, residual projections scaled by `1/sqrt(2·n_layers)`)
matches `zoology/model.py`.

**State size** (the x-axis quantity) mirrors `LanguageModel.state_size`:
each `MHA` layer contributes `2 · d_model · seq_len` (keys + values cached over the
sequence), summed over layers and multiplied by 4 bytes (float32).

In [ ]:
class SelfAttention(nn.Module):
    """Causal softmax attention (zoology.mixers.attention.SelfAttention)."""
    def __init__(self, attention_dropout: float = 0.0):
        super().__init__()
        self.dropout_p = attention_dropout

    def forward(self, qkv):
        seqlen = qkv.shape[1]
        q, k, v = qkv.unbind(dim=2)
        softmax_scale = 1.0 / math.sqrt(q.shape[-1])
        scores = torch.einsum("bthd,bshd->bhts", q, k * softmax_scale)
        causal_mask = torch.triu(torch.full((seqlen, seqlen), -10000.0, device=scores.device), 1)
        scores = scores + causal_mask.to(dtype=scores.dtype)
        attention = torch.softmax(scores, dim=-1, dtype=v.dtype)
        attention = F.dropout(attention, self.dropout_p if self.training else 0.0)
        out = torch.einsum("bhts,bshd->bthd", attention, v)
        return out


class MHA(nn.Module):
    """Multi-head self-attention (zoology.mixers.attention.MHA)."""
    def __init__(self, d_model: int, num_heads: int = 1, bias: bool = True, dropout: float = 0.0, layer_idx: int = None):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.Wqkv = nn.Linear(d_model, 3 * d_model, bias=bias)
        self.inner_attn = SelfAttention(attention_dropout=dropout)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        qkv = self.Wqkv(x)
        qkv = rearrange(qkv, "... (three h d) -> ... three h d", three=3, d=self.head_dim)
        context = self.inner_attn(qkv)
        return self.out_proj(rearrange(context, "... h d -> ... (h d)"))

    def state_size(self, sequence_length: int):
        # KV cache: keys + values over the sequence
        return 2 * self.d_model * sequence_length


class MLP(nn.Module):
    """Feed-forward state-mixer (zoology.mixers.mlp.MLP)."""
    def __init__(self, d_model: int, hidden_mult: int = 4, activation=F.gelu):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_model * hidden_mult)
        self.activation = activation
        self.fc2 = nn.Linear(d_model * hidden_mult, d_model)

    def forward(self, x):
        return self.fc2(self.activation(self.fc1(x)))


class TransformerBlock(nn.Module):
    """Pre-norm block: norm -> MHA, then norm -> MLP (zoology.model.TransformerBlock)."""
    def __init__(self, cfg: ModelConfig, layer_idx: int):
        super().__init__()
        self.sequence_mixer = MHA(cfg.d_model, num_heads=cfg.num_heads, dropout=cfg.attn_dropout, layer_idx=layer_idx)
        self.state_mixer = MLP(cfg.d_model, hidden_mult=cfg.mlp_hidden_mult)
        self.dropout1 = nn.Dropout(cfg.embed_dropout if layer_idx == 0 else cfg.resid_dropout)
        self.norm1 = nn.LayerNorm(cfg.d_model, eps=cfg.layer_norm_epsilon)
        self.dropout2 = nn.Dropout(cfg.resid_dropout)
        self.norm2 = nn.LayerNorm(cfg.d_model, eps=cfg.layer_norm_epsilon)

    def forward(self, hidden_states, residual=None):
        dropped = self.dropout1(hidden_states)
        residual = (dropped + residual) if residual is not None else dropped
        hidden_states = self.norm1(residual)
        hidden_states = self.sequence_mixer(hidden_states)

        dropped = self.dropout2(hidden_states)
        residual = dropped + residual
        hidden_states = self.norm2(residual)
        hidden_states = self.state_mixer(hidden_states)
        return hidden_states, residual


class TokenEmbeddings(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.word_embeddings = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if not cfg.learnable_word_embeddings:
            self.word_embeddings.weight.requires_grad = False
        self.max_position_embeddings = cfg.max_position_embeddings
        if self.max_position_embeddings > 0:
            self.position_embeddings = nn.Embedding(cfg.max_position_embeddings, cfg.d_model)

    def forward(self, input_ids):
        emb = self.word_embeddings(input_ids)
        if self.max_position_embeddings > 0:
            pos = torch.arange(input_ids.shape[1], dtype=torch.long, device=input_ids.device)
            emb = emb + self.position_embeddings(pos)
        return emb


def _init_weights(module, n_layers, initializer_range=0.02):
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, std=initializer_range)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        if not getattr(module, "_no_reinit", False):
            nn.init.normal_(module.weight, std=initializer_range)
    # GPT-2-style residual rescaling for projections feeding the residual stream
    for name, p in module.named_parameters():
        if name in ("out_proj.weight", "fc2.weight"):
            nn.init.normal_(p, mean=0.0, std=initializer_range / math.sqrt(2 * n_layers))


class LanguageModel(nn.Module):
    """Mirrors zoology.model.LanguageModel for block_type='TransformerBlock'."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.embeddings = TokenEmbeddings(cfg)
        self.layers = nn.ModuleList([TransformerBlock(cfg, layer_idx=i) for i in range(cfg.n_layers)])
        self.drop_f = nn.Dropout(cfg.resid_dropout)
        self.ln_f = nn.LayerNorm(cfg.d_model, eps=cfg.layer_norm_epsilon)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        from functools import partial
        self.apply(partial(_init_weights, n_layers=cfg.n_layers, initializer_range=cfg.initializer_range))

        if cfg.learnable_word_embeddings:  # weight tying
            self.lm_head.weight = self.embeddings.word_embeddings.weight

    def forward(self, input_ids):
        hidden_states = self.embeddings(input_ids)
        residual = None
        for layer in self.layers:
            hidden_states, residual = layer(hidden_states, residual)
        dropped = self.drop_f(hidden_states)
        residual = dropped + residual
        hidden_states = self.ln_f(residual)
        return self.lm_head(hidden_states)

    def state_size(self, sequence_length: int) -> int:
        # zoology convention: sum per-layer KV state, x4 bytes (float32)
        total = sum(layer.sequence_mixer.state_size(sequence_length) for layer in self.layers)
        return 4 * total


def attention_state_size(d_model: int, n_layers: int, sequence_length: int) -> int:
    """Closed form of LanguageModel.state_size for attention: 4 * n_layers * 2 * d_model * L."""
    return 4 * n_layers * 2 * d_model * sequence_length


def d_model_for_state_size(target_state: float, n_layers: int, sequence_length: int, num_heads: int) -> int:
    """Invert the state-size formula to pick the d_model that lands nearest `target_state`."""
    raw = target_state / (4 * n_layers * 2 * sequence_length)
    d = max(num_heads, int(round(raw)))
    if d % num_heads != 0:           # keep divisible by num_heads
        d += num_heads - (d % num_heads)
    return d

## 4. Training loop
Based on `zoology/train.py`: AdamW, cross-entropy with `ignore_index=-100` (so only
answer positions count), per-example recall accuracy over non-ignored positions. On top
of the zoology defaults we add three things that fix the wide-model overfitting:
* **Per-step LR schedule:** linear **warmup over 1 epoch**, then cosine decay to 0
  (also smooths the per-epoch loss sawtooth). The peak LR is the width-scaled
  `lr_for_d_model`.
* **Gradient clipping** at `max_norm=1.0` to tame the wide model's early steps.
* **Early stopping** that fires either when the task is solved (`valid/accuracy > 0.99`)
  **or** when `valid/accuracy` hasn't improved for `patience=10` epochs (so an
  overfitting model stops instead of running its val loss up). We report the **best**
  `valid/accuracy`, which is what the final plot uses.

In [ ]:
def compute_metrics(preds, targets, ignore_index=-100):
    """Per-example recall accuracy over non-ignored positions (zoology.train.compute_metrics)."""
    accs = []
    for pred, target in zip(preds, targets):
        mask = target != ignore_index
        accs.append((pred[mask] == target[mask]).float().mean().item())
    return accs


def train_one_run(model: LanguageModel, train_dl, test_dl, logger, peak_lr: float):
    model.to(DEVICE)
    loss_fn = nn.CrossEntropyLoss()  # default ignore_index = -100
    optimizer = torch.optim.AdamW(model.parameters(), lr=peak_lr, weight_decay=TRAIN.weight_decay)

    # Per-step schedule: linear warmup over `warmup_epochs`, then cosine decay to 0.
    steps_per_epoch = len(train_dl)
    warmup_steps = max(1, int(TRAIN.warmup_epochs * steps_per_epoch))
    total_steps = max(warmup_steps + 1, TRAIN.max_epochs * steps_per_epoch)

    def lr_lambda(step):
        if step < warmup_steps:                       # linear 0 -> 1
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)  # cosine 1 -> 0
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_acc, epochs_no_improve = 0.0, 0
    final_metrics = {}
    for epoch in range(TRAIN.max_epochs):
        # ---- train ----
        model.train()
        for inputs, targets in tqdm(train_dl, desc=f"train {epoch+1}/{TRAIN.max_epochs}", leave=False):
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            logits = model(inputs)
            loss = loss_fn(rearrange(logits, "... c -> (...) c"), targets.flatten())
            loss.backward()
            if TRAIN.grad_clip and TRAIN.grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), TRAIN.grad_clip)
            optimizer.step()
            scheduler.step()
            logger.log({"train/loss": loss.item(), "lr": scheduler.get_last_lr()[0], "epoch": epoch})

        # ---- eval ----
        model.eval()
        test_loss, accs = 0.0, []
        with torch.no_grad():
            for inputs, targets in test_dl:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                logits = model(inputs)
                loss = loss_fn(rearrange(logits, "... c -> (...) c"), targets.flatten())
                test_loss += loss.item() / len(test_dl)
                accs.extend(compute_metrics(logits.argmax(-1).cpu(), targets.cpu()))

        acc = float(np.mean(accs))
        if acc > best_acc:
            best_acc, epochs_no_improve = acc, 0
        else:
            epochs_no_improve += 1
        final_metrics = {"valid/loss": test_loss, "valid/accuracy": acc, "valid/best_accuracy": best_acc}
        logger.log({"epoch": epoch, **final_metrics})
        print(f"  epoch {epoch+1:>3}/{TRAIN.max_epochs}  valid/loss={test_loss:.4f}  "
              f"valid/accuracy={acc:.4f}  best={best_acc:.4f}  no_improve={epochs_no_improve}")

        # ---- early stopping: solved, or no val-accuracy improvement for `patience` epochs ----
        if acc > TRAIN.early_stopping_threshold:
            print(f"  early stop (solved): valid/accuracy {acc:.4f} > {TRAIN.early_stopping_threshold}")
            break
        if epochs_no_improve >= TRAIN.patience:
            print(f"  early stop (plateau): no valid/accuracy improvement for {TRAIN.patience} "
                  f"epochs (best={best_acc:.4f})")
            break

    final_metrics["valid/best_accuracy"] = best_acc
    return final_metrics

## 5. Weights & Biases setup
We log in (skipped if `WANDB_MODE` is `disabled`/`offline`). The helper below assembles
the **complete nested config** — task, model, training, derived state size, seed, and
the runtime/library environment — which is what gets stored in W&B.

In [ ]:
import wandb

if WANDB_MODE not in ("disabled", "offline"):
    try:
        wandb.login()
    except Exception as e:
        print(f"wandb.login() failed ({e}); falling back to offline mode.")
        WANDB_MODE = "offline"


def build_full_config(model_cfg: ModelConfig, state_size: int,
                      num_parameters: int, fingerprint: str, sweep_id: str, peak_lr: float) -> dict:
    """The full, reproducible config saved to W&B."""
    return {
        "architecture": "transformer",
        "task": asdict(TASK),
        "model": asdict(model_cfg),
        "train": asdict(TRAIN),
        "sweep": {
            "sweep_id": sweep_id,
            "d_models": list(D_MODELS),
            "n_points": N_POINTS,
            "lr_rule": {
                "type": "clamped_inverse_width",
                "lr_ref": LR_REF, "d_ref": D_REF, "lr_min": LR_MIN, "lr_max": LR_MAX,
            },
        },
        "derived": {
            "state_size": state_size,
            "state_size_seq_len": TASK.input_seq_len,
            "num_parameters": num_parameters,
            "peak_learning_rate": peak_lr,
        },
        "reproducibility": {
            "seed": SEED,
            "dataset_fingerprint": fingerprint,
            "torch_version": torch.__version__,
            "numpy_version": np.__version__,
            "python_version": platform.python_version(),
            "device": DEVICE,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "cudnn_deterministic": True,
        },
    }


class WandbLogger:
    """Thin logger mirroring zoology.logger.WandbLogger (no-ops if W&B is disabled)."""
    def __init__(self, run):
        self.run = run

    def log(self, metrics: dict):
        if self.run is not None:
            self.run.log(metrics)

## 6. Run the sweep
We train one transformer per width in `D_MODELS`. Each width maps to a state size (the
x-axis) via `attention_state_size`, and uses its own width-scaled peak learning rate
(`lr_for_d_model`, clamped 1/width) so the widest model trains gently enough to avoid
overfitting. Every run re-seeds from the same global `SEED` (so the dataset and init are
deterministic) and saves its full config to W&B.

In [ ]:
SWEEP_ID = uuid.uuid4().hex[:8]
GROUP = f"mqar-transformer-exp001b"

# n_layers is fixed by the (base) model config; each chosen width maps to a state size.
N_LAYERS = ModelConfig().n_layers
d_models = list(D_MODELS)
state_sizes = [attention_state_size(d, N_LAYERS, TASK.input_seq_len) for d in d_models]

print("Planned sweep (state size is the x-axis):")
for d, s in zip(d_models, state_sizes):
    print(f"  d_model={d:>4d}  ->  state_size={s:>11d}  ->  lr={lr_for_d_model(d):.2e}")

results = []
for i, d_model in enumerate(d_models):
    peak_lr = lr_for_d_model(d_model)
    print(f"\n=== Run {i+1}/{N_POINTS}: d_model={d_model} (lr={peak_lr:.2e}) ===")
    set_determinism(SEED)

    # data (identical task across runs) + model
    train_dl, test_dl, fingerprint = build_dataloaders(TASK, seed=SEED)
    model_cfg = ModelConfig(d_model=d_model, vocab_size=TASK.vocab_size,
                            max_position_embeddings=TASK.input_seq_len)
    model = LanguageModel(model_cfg)

    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    state_size = model.state_size(sequence_length=TASK.input_seq_len)
    full_config = build_full_config(model_cfg, state_size, num_params,
                                    fingerprint, SWEEP_ID, peak_lr)

    run = None
    if WANDB_MODE != "disabled":
        run = wandb.init(
            project=WANDB_PROJECT, entity=WANDB_ENTITY,
            name=f"d_model{d_model}-state{state_size}", group=GROUP, job_type="train",
            mode=WANDB_MODE, config=full_config, reinit=True,
            tags=["mqar", "transformer", "state-size-sweep"],
        )
        run.log({"state_size": state_size, "num_parameters": num_params})
        # also persist the full config as a downloadable artifact for full reproducibility
        with open("config.json", "w") as f:
            json.dump(full_config, f, indent=2)
        run.save("config.json")

    logger = WandbLogger(run)
    metrics = train_one_run(model, train_dl, test_dl, logger, peak_lr=peak_lr)

    if run is not None:
        run.summary.update({"state_size": state_size, "num_parameters": num_params,
                            "peak_learning_rate": peak_lr, **metrics})
        run.finish()

    results.append({
        "d_model": d_model,
        "state_size": int(state_size),
        "num_parameters": int(num_params),
        "learning_rate": peak_lr,
        "valid_accuracy": metrics["valid/accuracy"],
        "valid_best_accuracy": metrics["valid/best_accuracy"],
    })

print("\nSweep complete.")
for r in results:
    print(f"  state_size={r['state_size']:>11d}  d_model={r['d_model']:>4d}  "
          f"lr={r['learning_rate']:.2e}  best_acc={r['valid_best_accuracy']:.4f}")

## 7. The recreated plot: state size vs. recall accuracy
Accuracy on a linear y-axis against state size on a **log** x-axis — the canonical
zoology view. The figure is saved to disk, displayed inline, and logged to a dedicated
W&B **summary** run as a matplotlib image, a `wandb.Table`, and a native line plot.

In [ ]:
import matplotlib.pyplot as plt

results_sorted = sorted(results, key=lambda r: r["state_size"])
xs = [r["state_size"] for r in results_sorted]
ys = [r["valid_best_accuracy"] for r in results_sorted]   # best val accuracy (robust to overfitting tail)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(xs, ys, marker="o", linewidth=2, markersize=8, color="#3b76af", label="Transformer (attention)")
ax.set_xscale("log")
ax.set_xlabel("State size (bytes)")
ax.set_ylabel("MQAR recall accuracy (best)")
ax.set_ylim(-0.02, 1.02)
ax.set_title(f"MQAR: State Size vs. Recall Accuracy\n(Transformer, seq_len={TASK.input_seq_len}, "
             f"{TASK.num_kv_pairs} KV pairs, vocab={TASK.vocab_size})")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
for x, y, d in zip(xs, ys, [r["d_model"] for r in results_sorted]):
    ax.annotate(f"d={d}", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
fig.tight_layout()
fig.savefig("mqar_state_size_vs_accuracy.png", dpi=150)
plt.show()

if WANDB_MODE != "disabled":
    summary = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                         name=f"summary-{SWEEP_ID}", group=GROUP, job_type="summary",
                         mode=WANDB_MODE, reinit=True, tags=["mqar", "transformer", "summary"])
    table = wandb.Table(
        columns=["state_size", "valid_best_accuracy", "valid_accuracy", "d_model",
                 "learning_rate", "num_parameters"],
        data=[[r["state_size"], r["valid_best_accuracy"], r["valid_accuracy"], r["d_model"],
               r["learning_rate"], r["num_parameters"]] for r in results_sorted])
    summary.log({
        "state_size_vs_accuracy_plot": wandb.Image(fig),
        "results": table,
        "state_size_vs_accuracy": wandb.plot.line(table, "state_size", "valid_best_accuracy",
                                                  title="MQAR: state size vs recall accuracy"),
    })
    summary.finish()
    print("Logged summary plot + table to Weights & Biases.")

## 8. Notes & extensions
* **Expected shape.** Attention has effectively unbounded state, so accuracy rises with
  `d_model` (state size) and saturates near 1.0 once the model has enough capacity to
  separate the large vocabulary and route keys to values — the headline zoology result
  for transformers.
* **Learning rate.** We use a width-scaled peak LR (`lr_for_d_model`: clamped 1/width)
  so wider models train gently — this is what stops the largest model from overfitting.
  The paper instead picks the best LR per point from a grid (`np.logspace(-3, -1.5, 4)`);
  for the most faithful result, wrap the sweep loop in `for lr in ...` and keep the best.
* **Other architectures.** Swap `MHA` for a sub-quadratic mixer (Mamba, BASED, GLA, …)
  to reproduce the *recall-degradation* curves where bounded state caps accuracy.
* **Harder task.** Increase `num_kv_pairs` / `input_seq_len` in `TASK` to push the
  transition region.